# 04 extract study district

Splits the national files into the supply set and the candidate set for the
study district.

The district is the one identified in 03. Selection happens there because it
depends on comparing all districts; this notebook only slices.

Writes `kikuube_supply.csv` and `kikuube_candidates.csv`.

In [1]:
import sys
from pathlib import Path

# config.py sits beside the notebooks, so the working directory is enough. If a
# notebook is run from elsewhere, walk up until it is found.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
sys.path.insert(0, str(ROOT))
from config import *

import ast
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from matplotlib.lines import Line2D
from rasterio.features import shapes
from rasterio.mask import mask
from shapely.geometry import shape

## 1. Slice

In [2]:
x = pd.read_csv(ANALYSIS_SAMPLE, low_memory=False)
cand = pd.read_csv(CANDIDATES_NAT, low_memory=False)

ks = x[(x.district == DISTRICT.upper()) & (x["#status_id"] == "Yes")]
kc = cand[cand.district == DISTRICT.upper()]

print(f"{DISTRICT}")
print(f"  functioning supply points   {len(ks):,}")
print(f"  rehabilitation candidates   {len(kc):,}")

Kikuube
  functioning supply points   1,209
  rehabilitation candidates   320


## 2. Recorded label against geometry

The slice above uses the district name the exchange records. A spatial join to
the 2020 COD-AB boundaries returns the same points, because the district was
created in 2018 and its recent records come from a survey conducted on current
boundaries. The check is run once here and the recorded label is used
thereafter, which is what removes the need for a district boundary file
anywhere in the pipeline.

In [3]:
sub = gpd.read_file(SUBCOUNTY_GJ).to_crs(CRS_GEO)
outline = sub.dissolve().geometry.iloc[0]

def inside(df):
    pts = gpd.points_from_xy(df["#lon_deg"], df["#lat_deg"])
    return gpd.GeoSeries(pts, crs=CRS_GEO).within(outline).sum()

print("all points   by recorded label %5d   by geometry %5d"
      % (len(x[x.district == DISTRICT.upper()]), inside(x)))
print("candidates   by recorded label %5d   by geometry %5d"
      % (len(kc), inside(cand)))

all points   by recorded label  1733   by geometry  1733
candidates   by recorded label   320   by geometry   320


## 3. Capacity

The capacity field is what makes the capacitated formulation possible, and the
comparison below is what makes it necessary.

In [4]:
print(kc.usage_capacity.value_counts(dropna=False).to_string())
print(f"\ncapacity recorded for {kc.usage_capacity.notna().sum()} of {len(kc)}")
print(f"sum of candidate capacity {kc.usage_capacity.sum():,.0f} persons")

over = kc.local_population_1km > kc.usage_capacity
print(f"candidates with more people within 1 km than their capacity: "
      f"{over.sum()} of {len(kc)} ({over.mean() * 100:.1f} per cent)")
print(f"median population within 1 km {kc.local_population_1km.median():,.0f}, "
      f"median capacity {kc.usage_capacity.median():,.0f}")

usage_capacity
300.0    283
50.0      19
250.0     18

capacity recorded for 320 of 320
sum of candidate capacity 90,350 persons
candidates with more people within 1 km than their capacity: 314 of 320 (98.1 per cent)
median population within 1 km 1,237, median capacity 300


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))

tech = kc["#water_tech_category"].fillna("Not recorded").value_counts()
tech.plot(kind="bar", ax=axes[0], color="#6baed6", width=0.62)
axes[0].set_title("(a) Technology", fontsize=11)
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=0)

cap = kc.usage_capacity.value_counts().sort_index()
cap.index = [f"{int(v)}" for v in cap.index]
cap.plot(kind="bar", ax=axes[1], color="#6baed6", width=0.62)
axes[1].set_title("(b) Usage capacity", fontsize=11)
axes[1].set_xlabel("Persons per point")
axes[1].tick_params(axis="x", rotation=0)

yr = pd.to_datetime(kc["#report_date"], errors="coerce").dt.year
yc = yr.value_counts().sort_index()
yc.index = [f"{int(v)}" for v in yc.index]
yc.plot(kind="bar", ax=axes[2], color="#6baed6", width=0.62)
axes[2].set_title("(c) Year of record", fontsize=11)
axes[2].set_xlabel("")
axes[2].tick_params(axis="x", rotation=0)

# Counts above each bar, so the reader does not have to trace the gridline.
for ax in axes:
    for patch in ax.patches:
        ax.annotate(f"{int(patch.get_height())}",
                    (patch.get_x() + patch.get_width() / 2, patch.get_height()),
                    ha="center", va="bottom", fontsize=8.5, xytext=(0, 2),
                    textcoords="offset points")
    ax.set_ylabel("Candidates", fontsize=10)
    ax.spines[["top", "right"]].set_visible(False)
    ax.margins(y=0.14)

plt.tight_layout()
plt.savefig(FIG / "fig_candidates_profile.png", dpi=300, bbox_inches="tight")
plt.show()

## 4. Write

In [6]:
ks.to_csv(KIKUUBE_SUPPLY, index=False)
kc.to_csv(KIKUUBE_CAND, index=False)
print("written")

written


In [ ]:
sub = gpd.read_file(SUBCOUNTY_GJ).to_crs(CRS_GEO)
sub["subcounty"] = sub.Sub_County.str.strip().str.upper()

# Census population per sub-county, with the refugee settlement already merged
# into its host in 02, so the seven polygons and the seven counts correspond.
wts = pd.read_csv(SUBCOUNTY_WTS)
sub = sub.merge(wts[["subcounty", "population"]], on="subcounty",
                validate="one_to_one").rename(columns={"population": "census_pop"})

# Water bodies as polygons, from the friction surface (~0.060 min/m = 1 km/h)
with rasterio.open(FRICTION_TIF) as src:
    fr_arr, fr_T = mask(src, sub.dissolve().geometry, crop=True,
                        nodata=np.nan, filled=True)
fr = fr_arr[0]
water_mask = np.isclose(fr, 0.060, atol=0.002).astype("uint8")
polys = [shape(g) for g, v in shapes(water_mask, mask=water_mask.astype(bool),
                                     transform=fr_T) if v == 1]
water_gdf = gpd.GeoDataFrame(geometry=polys, crs=CRS_GEO)

# NOTE: use a column name that does not collide with GeoDataFrame's built-in
# .area property, which computes on the geographic CRS and gives nonsense.
water_gdf["area_m2"] = water_gdf.to_crs(CRS_METRIC).geometry.area

# Specks below 5 ha are raster artefacts rather than water bodies at this scale.
MIN_AREA_M2 = 50_000
water_gdf = water_gdf[water_gdf["area_m2"] > MIN_AREA_M2]

# Smooth the raster-derived staircase edges: dissolve adjacent cells into
# continuous shapes, then simplify. A positive buffer round-trip rounds inside
# and outside corners; simplify removes the remaining pixel steps. The radius
# has to exceed the cell width to have any effect, and much above this the
# shoreline inlets start to disappear.
SMOOTH_M = 120   # friction cells are ~93 m
water_smooth = gpd.GeoDataFrame(
    geometry=[water_gdf.geometry.union_all()], crs=CRS_GEO).to_crs(CRS_METRIC)
water_smooth["geometry"] = (water_smooth.geometry
                            .buffer(SMOOTH_M).buffer(-SMOOTH_M)
                            .simplify(SMOOTH_M / 3))
water_smooth = water_smooth.to_crs(CRS_GEO).explode(index_parts=False)
water_smooth = water_smooth[
    water_smooth.to_crs(CRS_METRIC).geometry.area > MIN_AREA_M2]

print(f"water polygons after cleaning: {len(water_gdf)} raw -> "
      f"{len(water_smooth)} smoothed, "
      f"total area {water_gdf['area_m2'].sum() / 1e6:.1f} km2")

# Panel (a) locates the district nationally. The outline is dissolved from the
# COD-AB sub-county layer already in use, so no extra boundary file is needed.
national = gpd.read_file(COD_AB_SUBCOUNTIES_GJ).to_crs(CRS_GEO)
country = gpd.GeoDataFrame(geometry=[national.geometry.union_all()], crs=CRS_GEO)
district = gpd.GeoDataFrame(geometry=[sub.geometry.union_all()], crs=CRS_GEO)

fig = plt.figure(figsize=(11.4, 6.2))
gs = fig.add_gridspec(1, 2, width_ratios=[1, 3.1], wspace=0.02)

axl = fig.add_subplot(gs[0])
country.plot(ax=axl, color="#f0f0f0", edgecolor="#999999", linewidth=0.6)
district.plot(ax=axl, color="#08519c", edgecolor="black", linewidth=0.5)
axl.set_title("(a) Location in Uganda", fontsize=10, loc="left")
axl.set_axis_off()

ax = fig.add_subplot(gs[1])

sub.plot(column="census_pop", ax=ax, cmap="Greens", alpha=0.55,
         edgecolor="#999999", linewidth=0.9, zorder=1, legend=True,
         legend_kwds={"label": "subcounty population (demand)", "shrink": 0.6})

if len(water_smooth):
    water_smooth.plot(ax=ax, color="#3182bd", edgecolor="#08306b",
                      linewidth=0.8, alpha=0.95, zorder=3)

for _, r in sub.iterrows():
    c = r.geometry.representative_point()
    ax.annotate(r.Sub_County, (c.x, c.y), fontsize=6.5, color="#333",
                ha="center", va="center", zorder=6,
                bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none",
                          alpha=0.65))

# Functioning points: a single class, distinguished by marker shape rather than
# by colour, so the capacity ramp below reads on its own.
ax.scatter(ks["#lon_deg"], ks["#lat_deg"], s=14, c="#ffe14d", marker="h",
           edgecolor="#7a6500", linewidth=0.3, zorder=4)

# Candidates: capacity is ordinal, so the three classes take an ordered ramp
# from orange through red to purple. Size is held nearly constant so that
# colour carries the distinction and dense clusters stay legible.
CAP_COLOUR = {300: "#7b3294", 250: "#d7191c", 50: "#e07000"}
CAP_SIZE = {300: 26, 250: 18, 50: 11}

cap = pd.to_numeric(kc.usage_capacity, errors="coerce")
for c in (50, 250, 300):                     # largest drawn last, on top
    m = cap == c
    if m.any():
        ax.scatter(kc.loc[m, "#lon_deg"], kc.loc[m, "#lat_deg"],
                   s=CAP_SIZE[c], c=CAP_COLOUR[c], marker="o",
                   edgecolor="black", linewidth=0.25, alpha=0.9, zorder=5)

other = cap.isna() | ~cap.isin(list(CAP_COLOUR))
if other.any():
    ax.scatter(kc.loc[other, "#lon_deg"], kc.loc[other, "#lat_deg"],
               s=11, c="#999999", marker="o", edgecolor="black",
               linewidth=0.25, alpha=0.9, zorder=5)

cap_handles = [Line2D([0], [0], marker="o", color="w",
                      markerfacecolor=CAP_COLOUR[c], markeredgecolor="black",
                      markeredgewidth=0.25, markersize=np.sqrt(CAP_SIZE[c]) * 1.6,
                      linestyle="none", label=f"capacity {c}")
               for c in (300, 250, 50)]
leg1 = ax.legend(handles=cap_handles, loc="lower left", fontsize=7, frameon=False,
                 title="candidate capacity", title_fontsize=7, labelspacing=0.8)
ax.add_artist(leg1)

ax.legend(handles=[
    Line2D([0], [0], marker="h", color="w", markerfacecolor="#ffd400",
           markeredgecolor="#7a6500", markersize=6, linestyle="none",
           label=f"functioning ({len(ks):,})"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#d7191c",
           markeredgecolor="black", markersize=5, linestyle="none",
           label=f"rehabilitation candidates ({len(kc)})"),
    Line2D([0], [0], marker="s", color="w", markerfacecolor="#3182bd",
           markeredgecolor="#08306b", markersize=8, linestyle="none",
           label="water body"),
], loc="upper right", fontsize=7, frameon=False)

fig.axes[-1].tick_params(labelsize=7)
fig.axes[-1].yaxis.label.set_size(7)

ax.set_title(f"(b) {DISTRICT} District", fontsize=10, loc="left")
ax.set_axis_off()
plt.tight_layout()
plt.savefig(FIG / "fig_study_area.png", dpi=300, bbox_inches="tight")
plt.show()

Expect 1,209 functioning points and 320 candidates.